# Granite Peak CRE: 04 Semantic layer (runs after the pipeline)

Builds what Genie and the app query on top of the gold tables:

1. **Metric views:** `portfolio_metrics` (ABR, WALT, rollover, abstraction coverage) and `occupancy_metrics` (occupancy rate by month, region and market). Definitions are in `src/governance/*.yaml`.
2. **Trusted H3 functions:** `assets_near_deal`, `assets_near` and `competitors_within`, defined in `src/genie/functions.sql`.
3. **Governed tags** on PII and confidential columns.
4. **Checks:** governance (row filter + masks), metric view results, the H3 functions, and Unity Catalog lineage from lease PDF to gold.

In [1]:
dbutils.widgets.text("catalog", "smmorgan_catalog", "Catalog")
dbutils.widgets.text("schema", "granite_commercial_real_estate", "Schema")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
uc_schema = f"{catalog}.{schema}"

import os, re
SRC = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
def read_src(rel):
    return open(os.path.join(SRC, rel)).read().replace("${uc_schema}", uc_schema)
print(f"Target: {uc_schema}  |  Source files: {SRC}")

Target: smmorgan_catalog.granite_commercial_real_estate  |  Source files: <user-workspace>/.bundle/granite_commercial_real_estate/dev/files/src


## 1. Metric views

In [2]:
for name in ["portfolio_metrics", "occupancy_metrics"]:
    yaml_text = read_src(f"governance/{name}.yaml")
    spark.sql(f"CREATE OR REPLACE VIEW {uc_schema}.{name} WITH METRICS LANGUAGE YAML AS $$\n{yaml_text}\n$$")
    print(f"created metric view {uc_schema}.{name}")

display(spark.sql(f"""
SELECT `fund_id`, MEASURE(abr) AS abr, MEASURE(lease_count) AS leases, round(MEASURE(walt_years), 2) AS walt_years,
       round(MEASURE(pct_abr_expiring_24m), 3) AS pct_abr_expiring_24m,
       round(MEASURE(abstraction_coverage_abr), 3) AS abstraction_coverage_abr
FROM {uc_schema}.portfolio_metrics GROUP BY ALL ORDER BY fund_id
"""))

created metric view smmorgan_catalog.granite_commercial_real_estate.portfolio_metrics
created metric view smmorgan_catalog.granite_commercial_real_estate.occupancy_metrics


fund_id    | abr                | leases | walt_years | pct_abr_expiring_24m | abstraction_coverage_abr
-----------+--------------------+--------+------------+----------------------+-------------------------
GP-FND-001 | 73073963.1         | 80     | 3.93       | 0.273                | 0.428                   
GP-FND-002 | 112573323.15999997 | 106    | 4.24       | 0.23                 | 0.324                   
GP-FND-003 | 84398886.89000002  | 94     | 3.58       | 0.306                | 0.502                   
GP-FND-004 | 100763213.26999995 | 111    | 4.2        | 0.18                 | 0.431                   


In [3]:
display(spark.sql(f"""
SELECT region, period_year, round(MEASURE(occupancy_rate), 4) AS occupancy_rate
FROM {uc_schema}.occupancy_metrics GROUP BY ALL ORDER BY region, period_year
"""))

region            | period_year | occupancy_rate
------------------+-------------+---------------
Mountain          | 2024        | 0.6914        
Mountain          | 2025        | 0.7414        
Mountain          | 2026        | 0.8336        
Pacific Northwest | 2024        | 0.6845        
Pacific Northwest | 2025        | 0.7173        
Pacific Northwest | 2026        | 0.7952        
Southwest         | 2024        | 0.6876        
Southwest         | 2025        | 0.733         
Southwest         | 2026        | 0.8354        


## 2. Trusted H3 functions

In [4]:
for chunk in re.split(r";\s*\n", read_src("genie/functions.sql")):
    body = "\n".join(l for l in chunk.splitlines() if not l.strip().startswith("--")).strip().rstrip(";")
    if body:
        spark.sql(body)
        print("created", re.search(r"FUNCTION\s+(\S+)\(", body).group(1))

created smmorgan_catalog.granite_commercial_real_estate.assets_near_deal
created smmorgan_catalog.granite_commercial_real_estate.assets_near
created smmorgan_catalog.granite_commercial_real_estate.competitors_within


In [5]:
print("Held assets within ~2 km of deal GP-ACQ-014:")
display(spark.sql(f"SELECT * FROM {uc_schema}.assets_near_deal('GP-ACQ-014', 7) ORDER BY distance_km"))

Held assets within ~2 km of deal GP-ACQ-014:


deal_id    | property_id  | property_name              | property_type | msa      | distance_km | occupancy_rate     | annual_base_rent  
-----------+--------------+----------------------------+---------------+----------+-------------+--------------------+-------------------
GP-ACQ-014 | GP-PROP-0077 | Granite Portland Retail 77 | Retail        | Portland | 0.8         | 1.0                | 1501465.4         
GP-ACQ-014 | GP-PROP-0089 | Granite Portland Office 89 | Office        | Portland | 1.12        | 1.0                | 1539424.4899999998
GP-ACQ-014 | GP-PROP-0085 | Granite Portland Retail 85 | Retail        | Portland | 1.14        | 0.7325361437643562 | 460595.76         
GP-ACQ-014 | GP-PROP-0081 | Granite Portland Retail 81 | Retail        | Portland | 1.31        | 1.0                | 1944771.9000000001
GP-ACQ-014 | GP-PROP-0087 | Granite Portland Office 87 | Office        | Portland | 1.8         | 0.8580155961539582 | 4972869.59        
GP-ACQ-014 | GP-PROP-0080 | Granit

In [6]:
first_retail = spark.sql(f"SELECT min(property_id) FROM {uc_schema}.property_h3 WHERE property_type = 'Retail'").first()[0]
print(f"POIs within ~1 km of {first_retail}:")
display(spark.sql(f"SELECT * FROM {uc_schema}.competitors_within('{first_retail}', 3) ORDER BY poi_count DESC"))

POIs within ~1 km of GP-PROP-0004:


property_id  | poi_group | poi_count | is_competitor
-------------+-----------+-----------+--------------
GP-PROP-0004 | amenity   | 1         | False        


## 3. Governed tags

In [7]:
TAGS = [
    ("tenant_directory", "contact_name",  "class.name"),
    ("tenant_directory", "contact_email", "class.email_address"),
    ("tenant_directory", "contact_phone", "class.phone_number"),
]
for table, column, tag in TAGS:
    try:
        spark.sql(f"ALTER TABLE {uc_schema}.{table} ALTER COLUMN {column} SET TAGS ('{tag}')")
        print(f"tagged {table}.{column} with {tag}")
    except Exception as e:
        print(f"could not tag {table}.{column}: {str(e).splitlines()[0][:160]}")
for table in ["rent_roll", "lease_abstract", "tenant_directory", "lease_discrepancies"]:
    try:
        spark.sql(f"ALTER TABLE {uc_schema}.{table} SET TAGS ('classification' = 'confidential')")
        print(f"tagged {table} classification=confidential")
    except Exception as e:
        print(f"could not tag {table}: {str(e).splitlines()[0][:160]}")

tagged tenant_directory.contact_name with class.name
tagged tenant_directory.contact_email with class.email_address
tagged tenant_directory.contact_phone with class.phone_number
tagged rent_roll classification=confidential
tagged lease_abstract classification=confidential
tagged tenant_directory classification=confidential
tagged lease_discrepancies classification=confidential


## 4. Governance checks

In [8]:
print("Tenant contact columns are masked for callers outside granite_pii_readers:")
display(spark.sql(f"SELECT tenant_id, tenant_name, contact_name, contact_email, contact_phone FROM {uc_schema}.tenant_directory ORDER BY tenant_id LIMIT 5"))
print("Row filter and masks attached to the gold tables:")
display(spark.sql(f"""
SELECT table_name, 'row filter' AS policy, filter_name AS function_name, target_columns AS columns
FROM {catalog}.information_schema.row_filters WHERE table_schema = '{schema}'
UNION ALL
SELECT table_name, 'column mask', mask_name, column_name
FROM {catalog}.information_schema.column_masks WHERE table_schema = '{schema}'
ORDER BY table_name, policy
"""))

Tenant contact columns are masked for callers outside granite_pii_readers:


tenant_id    | tenant_name        | contact_name | contact_email | contact_phone
-------------+--------------------+--------------+---------------+--------------
GP-TNT-00001 | Copper Financial   | [REDACTED]   | [REDACTED]    | [REDACTED]   
GP-TNT-00002 | Pine Legal Group   | [REDACTED]   | [REDACTED]    | [REDACTED]   
GP-TNT-00003 | Cedar Solutions    | [REDACTED]   | [REDACTED]    | [REDACTED]   
GP-TNT-00004 | Aspen Technologies | [REDACTED]   | [REDACTED]    | [REDACTED]   
GP-TNT-00005 | Eagle Financial    | [REDACTED]   | [REDACTED]    | [REDACTED]   


Row filter and masks attached to the gold tables:


table_name        | policy      | function_name                                                   | columns      
------------------+-------------+-----------------------------------------------------------------+--------------
occupancy_monthly | row filter  | smmorgan_catalog.granite_commercial_real_estate.fund_row_filter | fund_id      
rent_roll         | row filter  | smmorgan_catalog.granite_commercial_real_estate.fund_row_filter | fund_id      
rollover_schedule | row filter  | smmorgan_catalog.granite_commercial_real_estate.fund_row_filter | fund_id      
tenant_directory  | column mask | smmorgan_catalog.granite_commercial_real_estate.mask_contact    | contact_email
tenant_directory  | column mask | smmorgan_catalog.granite_commercial_real_estate.mask_contact    | contact_phone
tenant_directory  | column mask | smmorgan_catalog.granite_commercial_real_estate.mask_contact    | contact_name 


In [9]:
print("Unity Catalog lineage into the gold layer (system.access.table_lineage):")
display(spark.sql(f"""
SELECT DISTINCT source_table_name, target_table_name
FROM system.access.table_lineage
WHERE target_table_catalog = '{catalog}' AND target_table_schema = '{schema}'
  AND target_table_name IN ('silver_lease_text', 'silver_lease_extract', 'lease_abstract', 'lease_discrepancies', 'rent_roll')
  AND source_table_name IS NOT NULL
ORDER BY target_table_name, source_table_name
"""))

Unity Catalog lineage into the gold layer (system.access.table_lineage):


source_table_name    | target_table_name   
---------------------+---------------------
silver_lease_extract | lease_abstract      
silver_commlease     | lease_discrepancies 
silver_lease_extract | lease_discrepancies 
property_h3          | rent_roll           
silver_commlease     | rent_roll           
silver_lease_extract | rent_roll           
silver_property      | rent_roll           
silver_tenant        | rent_roll           
silver_unit          | rent_roll           
silver_lease_text    | silver_lease_extract
bronze_lease_docs    | silver_lease_text   
